# 🧠 STEMData — Construcción del modelo (versión definitiva)

Este cuaderno construye las capas del modelo, a partir del archivo ya limpio `stemdata_limpio.csv`, con las columnas: `municipio`, `zona`, `promedio_ciencias`, `promedio_matematicas`, `total_accesos_internet`.

- **Capa 1** — Brecha STEM (crítico/alto/medio/bajo)
- **Capa 2** — Viabilidad tecnológica: medición objetiva de conectividad (desconectado/básico/medio/alto), **sin mezclar la zona**
- **Capa 3** — Generador de planeación STEM+, donde la zona sí entra en juego (ajusta la logística de materiales, no el diagnóstico)
- **Visualización** — 2 gráficos que el docente ve dentro del agente, cada vez que consulta su municipio (no son gráficos generales para el pitch)

> 📌 La zona se retiró de la Capa 2 a propósito: confirmar la conectividad debe ser una medición objetiva de los datos, no un juicio pedagógico. Ese juicio (qué tan práctico es aplicar cierta actividad en zona rural vs. urbana) pertenece a la Capa 3, donde ahora sí se usa para ajustar la recomendación logística. La deserción escolar tampoco forma parte del modelo (se retiró al reemplazar el dataset del MEN por el de conectividad de la CRC, para mantener el proyecto dentro del límite de 2 datasets del concurso).

**Instrucciones:** ejecuta las celdas en orden, de arriba hacia abajo, con ▶ o `Shift + Enter`. Si en algún momento sale `NameError: name 'pd' is not defined`, la sesión se reinició — vuelve a ejecutar desde la Parte 1.


## Parte 1 — Cargar el archivo limpio

Si el archivo `stemdata_limpio.csv` no está ya en este Colab, primero súbelo con el botón de carpeta de la izquierda, o ejecuta la celda de subida que está comentada abajo.


In [ ]:
# Si necesitas subir el archivo desde tu computador, descomenta estas 2 líneas:
# from google.colab import files
# files.upload()

import pandas as pd

df = pd.read_csv("stemdata_limpio.csv")

print(f"Filas: {df.shape[0]} — Columnas: {df.shape[1]}")
df.head()

👉 Confirma que `df.columns` incluya: `municipio`, `zona`, `promedio_ciencias`, `promedio_matematicas`, `total_accesos_internet`.


In [ ]:
df.columns.tolist()

## Parte 2 — Capa 1: Brecha STEM

Combinamos los dos puntajes en un solo indicador, y lo comparamos contra el promedio nacional.


In [ ]:
df['puntaje_stem'] = (df['promedio_ciencias'] + df['promedio_matematicas']) / 2

df[['municipio', 'zona', 'puntaje_stem']].head()

### Calcular la brecha contra el promedio nacional

In [ ]:
promedio_nacional = df['puntaje_stem'].mean()
df['brecha_stem'] = df['puntaje_stem'] - promedio_nacional

print(f"Promedio nacional STEM: {promedio_nacional:.2f}")
df[['municipio', 'zona', 'puntaje_stem', 'brecha_stem']].head()

### Clasificar según qué tan lejos está del promedio (gravedad real, no un ranking artificial)

Usamos la desviación estándar en vez de dividir en 4 grupos de igual tamaño (`qcut`), porque queremos que el número de municipios en cada categoría refleje la gravedad real del problema, no una regla artificial de "25% cada uno".


In [ ]:
desviacion_stem = df['brecha_stem'].std()

def clasificar_brecha(brecha):
    if brecha < -desviacion_stem:
        return 'crítico'
    elif brecha < 0:
        return 'alto'
    elif brecha < desviacion_stem:
        return 'medio'
    else:
        return 'bajo'

df['nivel_brecha'] = df['brecha_stem'].apply(clasificar_brecha)

df['nivel_brecha'].value_counts()

## Parte 3 — Capa 2: Viabilidad tecnológica (medición objetiva de conectividad)

### Paso 3.1 — Clasificar conectividad por desviación estándar

Usamos `total_accesos_internet` (el total de accesos a internet fijo reportado en el municipio).


In [ ]:
promedio_conectividad = df['total_accesos_internet'].mean()
desviacion_conectividad = df['total_accesos_internet'].std()

def clasificar_conectividad(valor):
    if valor < promedio_conectividad - desviacion_conectividad:
        return 'desconectado'
    elif valor < promedio_conectividad:
        return 'básico'
    elif valor < promedio_conectividad + desviacion_conectividad:
        return 'medio'
    else:
        return 'alto'

df['nivel_conectividad_base'] = df['total_accesos_internet'].apply(clasificar_conectividad)

df[['municipio', 'zona', 'total_accesos_internet', 'nivel_conectividad_base']].head()

### Paso 3.2 — Revisar cómo aparece escrita la zona (útil para la Capa 3, más adelante)

In [ ]:
df['zona'].unique()

### Paso 3.3 — La conectividad queda como una medición objetiva, sin ajuste por zona

La zona (rural/urbana) no cambia el **dato** de conectividad — cambia qué tan realista es sugerir cierta actividad dado ese contexto, y eso le corresponde a la Capa 3 (el generador de planeación), no a esta capa de diagnóstico. Por eso `nivel_conectividad` es simplemente una copia directa de la clasificación por desviación estándar, sin ningún ajuste adicional.


In [ ]:
df['nivel_conectividad'] = df['nivel_conectividad_base']

df[['municipio', 'zona', 'total_accesos_internet', 'nivel_conectividad']].head(10)

### Paso 3.4 (opcional) — Árbol de decisión visual

Útil como evidencia visual de interpretabilidad para la sustentación. Usamos solo `total_accesos_internet`, ya que confirmamos que la zona no aporta información adicional para predecir el nivel de conectividad.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

X = df[['total_accesos_internet']]
y = df['nivel_conectividad']

arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X, y)

plt.figure(figsize=(14, 6))
plot_tree(arbol, feature_names=['total_accesos_internet'], class_names=arbol.classes_, filled=True)
plt.show()

## Parte 4 — Capa 3: Generador de planeación STEM+

Esta función recibe **municipio y zona**, y devuelve el diagnóstico completo más la planeación sugerida, combinando brecha STEM y viabilidad tecnológica.

Como esta herramienta la va a usar un docente escribiendo a mano, la función ignora tildes y acepta variantes de zona (ej. "urbana" o "urbano"), en vez de exigir el texto exacto.


In [ ]:
import unicodedata
import re

def quitar_tildes(texto):
    texto = str(texto).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r'[.,]', '', texto)   # quita puntos y comas (ej. "BOGOTA, D.C." -> "BOGOTA DC")
    texto = re.sub(r'\s+', ' ', texto)   # colapsa espacios dobles en uno
    return texto.strip()

def generar_planeacion(nombre_municipio, zona_deseada):
    municipio_buscado = quitar_tildes(nombre_municipio)
    zona_buscada = quitar_tildes(zona_deseada)

    # Acepta "URBANA", "URBANO", etc. -- se queda solo con la raíz "URBAN"
    if zona_buscada.startswith("URBAN"):
        zona_buscada = "URBANO"
    elif zona_buscada.startswith("RURAL"):
        zona_buscada = "RURAL"

    filtro = (
        (df['municipio'].apply(quitar_tildes) == municipio_buscado) &
        (df['zona'].apply(quitar_tildes) == zona_buscada)
    )
    fila = df[filtro]

    if fila.empty:
        return f"No se encontró '{nombre_municipio}' en zona '{zona_deseada}' en la base de datos."

    fila = fila.iloc[0]

    brecha = fila['nivel_brecha']
    conectividad = fila['nivel_conectividad']
    zona = fila['zona'].strip().upper()

    # La conectividad decide si la actividad puede apoyarse en herramientas digitales
    if conectividad in ['desconectado', 'básico']:
        actividad = "Actividad totalmente desconectada, con materiales físicos o reciclables."
    else:
        actividad = "Actividad híbrida, combinando materiales físicos con alguna herramienta digital básica."

    # La zona decide de dónde vienen esos materiales y cómo se organiza la logística
    if zona.startswith('RURAL'):
        logistica = "Prioriza materiales disponibles en el entorno inmediato (elementos del campo, reciclables, herramientas caseras), pensando en tiempos de desplazamiento y acceso limitado a papelerías o tiendas especializadas."
    else:
        logistica = "Pueden complementarse con materiales de papelería o ferretería cercana, y es más viable coordinar salidas cortas o recursos compartidos entre colegios de la misma zona."

    if brecha in ['crítico', 'alto']:
        enfoque = "Enfoque de refuerzo: actividad guiada paso a paso, con acompañamiento cercano del docente."
    else:
        enfoque = "Enfoque de profundización: actividad más abierta, con espacio para exploración autónoma."

    reporte = f"""
    📍 Municipio: {fila['municipio']} ({fila['zona']})
    📊 Nivel de brecha STEM: {brecha}
    📶 Nivel de conectividad: {conectividad}

    ✏️ Actividad sugerida: {actividad}
    🚚 Consideración logística: {logistica}
    🎯 Enfoque pedagógico sugerido: {enfoque}
    """
    return reporte

### Probarla con un municipio real

Ahora acepta el nombre con o sin tilde, y la zona en cualquier variante ("urbana", "urbano", "rural").


In [ ]:
print(generar_planeacion("MEDELLIN", "URBANA"))

In [ ]:
print(generar_planeacion("MEDELLIN", "RURAL"))

## Parte 5 — Visualización para el docente (dentro del agente)

Estos gráficos **no son para el equipo ni para el pitch** — son para que el docente los vea cada vez que consulta su municipio dentro del agente. Por eso, en vez de mostrar el país completo de una vez, cada gráfico **ubica al municipio consultado** dentro del panorama nacional, para que la persona entienda de un vistazo si está mejor o peor que el promedio.

Usamos **matplotlib**, la misma librería de siempre.


### Gráfico A — Tu municipio vs. el promedio nacional (puntaje STEM)

Un gráfico de barras simple con solo 2 barras: el puntaje del municipio consultado, y el promedio nacional, para comparar de un vistazo.


In [ ]:
def graficar_comparacion_stem(nombre_municipio, zona_deseada):
    municipio_buscado = quitar_tildes(nombre_municipio)
    zona_buscada = quitar_tildes(zona_deseada)
    if zona_buscada.startswith("URBAN"):
        zona_buscada = "URBANO"
    elif zona_buscada.startswith("RURAL"):
        zona_buscada = "RURAL"

    filtro = (
        (df['municipio'].apply(quitar_tildes) == municipio_buscado) &
        (df['zona'].apply(quitar_tildes) == zona_buscada)
    )
    fila = df[filtro]

    if fila.empty:
        print(f"No se encontró '{nombre_municipio}' en zona '{zona_deseada}'.")
        return

    puntaje_municipio = fila.iloc[0]['puntaje_stem']
    promedio_nacional_actual = df['puntaje_stem'].mean()

    etiquetas = [f"{nombre_municipio.title()}\n({zona_deseada.title()})", "Promedio\nnacional"]
    valores = [puntaje_municipio, promedio_nacional_actual]
    colores = ['#27ae60' if puntaje_municipio >= promedio_nacional_actual else '#c0392b', '#7f8c8d']

    plt.figure(figsize=(6, 5))
    plt.bar(etiquetas, valores, color=colores)
    plt.title('Tu municipio vs. el promedio nacional (puntaje STEM)')
    plt.ylabel('Puntaje STEM')
    plt.show()

graficar_comparacion_stem("MEDELLIN", "URBANA")

**Qué hace cada línea:**
- Repite la misma búsqueda de `generar_planeacion` (municipio + zona, ignorando tildes y variantes), para encontrar la fila correcta.
- `colores = [...]` decide el color de la primera barra: **verde** si el municipio está igual o por encima del promedio nacional, **rojo** si está por debajo — así el docente entiende el resultado con solo mirar el color, sin necesitar leer números.
- El resto dibuja las 2 barras, igual que en gráficos anteriores.


### Gráfico B — Dónde se ubica tu municipio en la distribución nacional de conectividad

Un histograma (barras que muestran cuántos municipios caen en cada rango de conectividad), con una línea vertical marcando exactamente dónde está el municipio consultado.


In [ ]:
def graficar_ubicacion_conectividad(nombre_municipio, zona_deseada):
    municipio_buscado = quitar_tildes(nombre_municipio)
    zona_buscada = quitar_tildes(zona_deseada)
    if zona_buscada.startswith("URBAN"):
        zona_buscada = "URBANO"
    elif zona_buscada.startswith("RURAL"):
        zona_buscada = "RURAL"

    filtro = (
        (df['municipio'].apply(quitar_tildes) == municipio_buscado) &
        (df['zona'].apply(quitar_tildes) == zona_buscada)
    )
    fila = df[filtro]

    if fila.empty:
        print(f"No se encontró '{nombre_municipio}' en zona '{zona_deseada}'.")
        return

    accesos_municipio = fila.iloc[0]['total_accesos_internet']

    plt.figure(figsize=(8, 5))
    plt.hist(df['total_accesos_internet'], bins=40, color='#bdc3c7')
    plt.axvline(accesos_municipio, color='#c0392b', linewidth=2.5, label=f"{nombre_municipio.title()}")
    plt.title('Dónde se ubica tu municipio en conectividad nacional')
    plt.xlabel('Total de accesos a internet')
    plt.ylabel('Cantidad de municipios')
    plt.xscale('log')
    plt.legend()
    plt.show()

graficar_ubicacion_conectividad("MEDELLIN", "URBANA")

**Qué hace cada línea:**
- `plt.hist(df['total_accesos_internet'], bins=40, ...)` dibuja el histograma: agrupa todos los municipios en 40 "cajones" según su nivel de conectividad, y la altura de cada barra es cuántos municipios caen ahí.
- `plt.axvline(accesos_municipio, ...)` dibuja una **línea vertical roja** exactamente en el valor del municipio consultado — así el docente ve de inmediato si su municipio está en la zona con pocos municipios (extremos) o en la zona más común (el centro).
- `plt.legend()` muestra la etiqueta de esa línea roja.


### Uniendo todo — la función que el agente llamaría en cada consulta

Esta es la función que combina el reporte de texto (Capa 3) con los 2 gráficos, todo en una sola consulta — esto es lo que el agente ejecutaría cada vez que un docente escribe su municipio.


In [ ]:
def consultar_municipio(nombre_municipio, zona_deseada):
    print(generar_planeacion(nombre_municipio, zona_deseada))
    graficar_comparacion_stem(nombre_municipio, zona_deseada)
    graficar_ubicacion_conectividad(nombre_municipio, zona_deseada)

consultar_municipio("MEDELLIN", "RURAL")

👉 Prueba `consultar_municipio(...)` con distintos municipios y zonas — esta es, en esencia, la experiencia completa que va a tener un docente al usar el agente: texto + los 2 gráficos, todo junto.


## Parte 6 — Agregar coordenadas y exportar datos para el mapa del prototipo

Ni Saber 11 ni el dataset de conectividad traen latitud/longitud. Para poder ubicar el municipio en un mapa dentro del prototipo, traemos una tabla de referencia oficial: **DIVIPOLA - Códigos municipios** (`gdxc-w37w`, datos.gov.co), usando su enlace de exportación directa (más confiable que el de consulta con parámetros, como confirmamos con el dataset de conectividad).

> 📌 Esta tabla es solo de **referencia geográfica** (para dibujar el mapa) — no es un dataset de análisis, así que no cuenta contra el límite de 1-2 datasets del concurso. Ese límite aplica a los datos que el modelo analiza (Saber 11 + conectividad), no a una tabla de coordenadas de apoyo visual.


In [ ]:
url_coordenadas = "https://www.datos.gov.co/api/views/gdxc-w37w/rows.csv?accessType=DOWNLOAD"

coordenadas_crudo = pd.read_csv(url_coordenadas)

print(f"Filas: {coordenadas_crudo.shape[0]} — Columnas: {coordenadas_crudo.shape[1]}")
coordenadas_crudo.columns.tolist()

👉 Revisa la lista de columnas que salió. En este dataset, la latitud y longitud vienen como **texto con coma decimal** (ej. `"-75,581775"`, al estilo de Excel en español), en vez de con punto — la siguiente celda ya corrige eso.


In [ ]:
COLUMNA_MUNICIPIO = "Nombre Municipio"
COLUMNA_LATITUD = "Latitud"
COLUMNA_LONGITUD = "longitud"

coordenadas = coordenadas_crudo[[COLUMNA_MUNICIPIO, COLUMNA_LATITUD, COLUMNA_LONGITUD]].copy()
coordenadas.columns = ["municipio", "latitud", "longitud"]

# Corregir la coma decimal (texto "-75,581775" -> número -75.581775)
coordenadas["latitud"] = coordenadas["latitud"].astype(str).str.replace(",", ".").astype(float)
coordenadas["longitud"] = coordenadas["longitud"].astype(str).str.replace(",", ".").astype(float)

coordenadas["municipio"] = coordenadas["municipio"].apply(quitar_tildes)
coordenadas["municipio"] = coordenadas["municipio"].replace({"BOGOTA DC": "BOGOTA"})
coordenadas = coordenadas.dropna().drop_duplicates(subset="municipio")

print(f"Municipios con coordenadas: {coordenadas.shape[0]}")
coordenadas.head()

### Cruzar las coordenadas con la tabla del modelo

In [ ]:
df['municipio_normalizado'] = df['municipio'].apply(quitar_tildes)

df_con_mapa = pd.merge(
    df,
    coordenadas,
    left_on='municipio_normalizado',
    right_on='municipio',
    how='left',
    suffixes=('', '_coord')
)

print(f"Filas con coordenadas encontradas: {df_con_mapa['latitud'].notna().sum()} de {df_con_mapa.shape[0]}")
df_con_mapa[['municipio', 'zona', 'latitud', 'longitud']].head(10)

👉 Si el número de "filas con coordenadas encontradas" es mucho menor al total, es la misma señal de siempre: nombres de municipio que no coinciden exactamente entre las dos fuentes. Puedes revisar con `df_con_mapa[df_con_mapa['latitud'].isna()]['municipio'].unique()` para ver cuáles quedaron sin coordenada.


### Exportar todo en un archivo JSON, listo para el prototipo

Este archivo tiene exactamente los datos que el prototipo necesita para dibujar el mapa y mostrar el detalle de cada municipio — en el mismo espíritu de los objetos `SABER11` y `CONEX` que ya existen en tu HTML, pero con datos reales y coordenadas incluidas.


In [ ]:
import json

registros = []
for _, fila in df_con_mapa.iterrows():
    if pd.isna(fila['latitud']):
        continue
    registros.append({
        "municipio": fila['municipio'],
        "zona": fila['zona'],
        "lat": round(float(fila['latitud']), 5),
        "lon": round(float(fila['longitud']), 5),
        "promedio_ciencias": round(float(fila['promedio_ciencias']), 1),
        "promedio_matematicas": round(float(fila['promedio_matematicas']), 1),
        "puntaje_stem": round(float(fila['puntaje_stem']), 1),
        "nivel_brecha": fila['nivel_brecha'],
        "total_accesos_internet": int(fila['total_accesos_internet']),
        "nivel_conectividad": fila['nivel_conectividad'],
    })

with open('stemdata_prototipo.json', 'w', encoding='utf-8') as f:
    json.dump(registros, f, ensure_ascii=False, indent=2)

print(f"Municipios exportados: {len(registros)}")

from google.colab import files
files.download('stemdata_prototipo.json')

## Cómo se conecta este archivo con el prototipo HTML

En tu archivo `STEMData_Prototipo__4_.html`, los objetos `SABER11` y `CONEX` están escritos a mano con municipios de ejemplo (Bogotá, Medellín, Cali...). El siguiente paso (fuera de este notebook, directamente en el HTML) es:

1. Agregar una etiqueta `<script src="stemdata_prototipo.json">` o cargar el archivo con `fetch('stemdata_prototipo.json')` en JavaScript.
2. Reemplazar los objetos `SABER11` y `CONEX` de ejemplo por los datos reales de este archivo.
3. Agregar el mapa con Leaflet.js (gratuito, sin necesidad de clave de API), usando los campos `lat` y `lon` de cada municipio para poner el marcador.

Te puedo ayudar con ese paso del HTML/JavaScript por separado, cuando quieras avanzar ahí.


## Parte 7 — Guardar el resultado final (CSV para el repositorio)

Este archivo ya trae todas las clasificaciones (`nivel_brecha`, `nivel_conectividad`) listas para conectar con el prototipo.


In [ ]:
df.to_csv('stemdata_con_modelo.csv', index=False, encoding='utf-8-sig')

from google.colab import files
files.download('stemdata_con_modelo.csv')

print("Archivo generado y descargado ✅")

## Qué hacer con este archivo

1. Busca `stemdata_con_modelo.csv` en tu carpeta de Descargas.
2. Súbelo a la carpeta `data/processed/` de tu repositorio en GitHub, junto a `stemdata_limpio.csv`.
3. Este archivo, junto con `generar_planeacion()`, `graficar_comparacion_stem()`, `graficar_ubicacion_conectividad()` y `consultar_municipio()`, son la base para conectar con `STEMData_Prototipo.html` — cada vez que un docente consulte su municipio, el prototipo debe llamar a estas mismas funciones para mostrarle el texto y los 2 gráficos juntos.
